### Collecting Deaths from their Lists

In [1]:
import os
import pandas as pd
import re
import warnings

from pyvis.network import Network

warnings.filterwarnings('ignore')

In [2]:
kills = pd.DataFrame(columns=['universe', 'character', 'killed', 'victims'])
deaths = pd.DataFrame(columns=['universe', 'entry', 'character', 'responsible party', 'death', 'type'])
edges = pd.DataFrame(columns=['universe', 'killer', 'victim', 'death', 'type'])

In [3]:
for path, subdirs, files in os.walk('Universes'):
	for name in files:
		if not name.endswith('.tsv'):
			continue

		file = os.path.join(path, name)
		parts = file.split('\\')

		universe = parts[1]
		entry = name.replace('.tsv', '')

		tsv = pd.read_csv(file, sep='\t')

		for i in range(len(tsv)):
			character = tsv['character'][i]
			killer = str(tsv['killed by'][i]).split(' | ')
			description = tsv['death'][i]
			type_var = tsv['type'][i]

			if killer == ['nan']:
				killer = []

			for i in range(len(killer)):
				row = [universe, killer[i], 1, character]
				kills.loc[len(kills)] = row

				row = [universe, killer[i], character, description, type_var]
				edges.loc[len(edges)] = row

			if killer == []:
				row = [universe, '', character, description, type_var]
				edges.loc[len(edges)] = row

			row = [universe, entry, character, killer, description, type_var]
			deaths.loc[len(deaths)] = row

### Replacing Groups with Character Names

In [4]:
groups = pd.read_csv('Database/group.tsv', sep='\t')
remove = False

rows = []
drops = []

for i in range(len(kills)):
	universe = kills['universe'][i]
	character = kills['character'][i]
	victim = kills['victims'][i]
	pool = groups[groups['universe'] == universe]
	pool.reset_index(inplace=True)

	for j in range(len(pool)):
		if character == pool['group'][j]:
			news = str(pool['characters'][j]).split(' | ')
			for new in news:
				rows.append({
					'universe': universe,
					'character': new,
					'killed': kills.loc[i, 'killed'],
					'victims': kills.loc[i, 'victims']
				})
			drops.append(i)

		if victim == pool['group'][j]:
			news = str(pool['characters'][j]).split(' | ')
			for new in news:
				rows.append({
					'universe': universe,
					'character': kills.loc[i, 'character'],
					'killed': kills.loc[i, 'killed'],
					'victims': new
				})
			drops.append(i)

kills.drop(index=drops, inplace=True)
kills = pd.concat([kills, pd.DataFrame(rows)], ignore_index=True)


for i in range(len(deaths)):
	universe = deaths['universe'][i]
	character = deaths['character'][i]
	party = deaths['responsible party'][i]
	pool = groups[groups['universe'] == universe]
	pool.reset_index(inplace=True)

	changes = []

	for j in range(len(pool)):
		if character == pool['group'][j]:
			new = str(pool['characters'][j]).split(' | ')
			deaths.at[i, 'character'] = new

	for member in party:
		remove = False
		for j in range(len(pool)):
			if member == pool['group'][j]:
				news = str(pool['characters'][j]).split(' | ')
				changes.extend(news)
				remove = True
		if not remove:
			changes.append(member)

	deaths.at[i, 'responsible party'] = changes


rows = []
drops = []

for i in range(len(edges)):
	universe = edges['universe'][i]
	killer = edges['killer'][i]
	victim = edges['victim'][i]
	pool = groups[groups['universe'] == universe]
	pool.reset_index(inplace=True)

	for j in range(len(pool)):
		if killer == pool['group'][j]:
			news = str(pool['characters'][j]).split(' | ')
			for new in news:
				rows.append({
					'universe': universe,
					'killer': new,
					'victim': edges.loc[i, 'victim'],
					'death': edges.loc[i, 'death'],
					'type': edges.loc[i, 'type']
				})
			drops.append(i)

		if victim == pool['group'][j]:
			news = str(pool['characters'][j]).split(' | ')
			for new in news:
				rows.append({
					'universe': universe,
					'killer': edges.loc[i, 'killer'],
					'victim': new,
					'death': edges.loc[i, 'death'],
					'type': edges.loc[i, 'type']
				})
			drops.append(i)

edges.drop(index=drops, inplace=True)
edges = pd.concat([edges, pd.DataFrame(rows)], ignore_index=True)

kills.reset_index(inplace=True)
edges.reset_index(inplace=True)

### Replacing Character Names with Aliases

In [5]:
aliases = pd.read_csv('Database/alias.tsv', sep='\t')
aliases['listed'] = aliases['character'].str.split(' / ')


for i in range(len(kills)):
	universe = kills['universe'][i]
	character = kills['character'][i]
	victim = kills['victims'][i]
	pool = aliases[aliases['universe'] == universe]
	pool.reset_index(inplace=True)

	for j in range(len(pool)):
		grouped = pool['listed'][j]
		if character in grouped:
			new = pool['character'][j]
			kills.at[i, 'character'] = new
		if victim in grouped:
			new = pool['character'][j]
			kills.at[i, 'victims'] = new


for i in range(len(deaths)):
	universe = deaths['universe'][i]
	character = deaths['character'][i]
	party = deaths['responsible party'][i]
	pool = aliases[aliases['universe'] == universe]
	pool.reset_index(inplace=True)

	for j in range(len(pool)):
		grouped = pool['listed'][j]
		if character in grouped:
			new = pool['character'][j]
			deaths.at[i, 'character'] = new

		changes = dict()
		for member in party:
			if member in grouped:
				new = pool['character'][j]
				changes[member] = new
		if changes:
			party = [changes.get(member, member) for member in party]
			deaths.at[i, 'responsible party'] = party


for i in range(len(edges)):
	universe = edges['universe'][i]
	killer = edges['killer'][i]
	victim = edges['victim'][i]
	pool = aliases[aliases['universe'] == universe]
	pool.reset_index(inplace=True)

	for j in range(len(pool)):
		grouped = pool['listed'][j]
		if killer in grouped:
			new = pool['character'][j]
			edges.at[i, 'killer'] = new
		if victim in grouped:
			new = pool['character'][j]
			edges.at[i, 'victim'] = new

### Analyzing and Organizing the Deaths

In [6]:
def count_and_name(name):
	if isinstance(name, str):
		match = re.match(r"^(<(\d+(?:[.,]\d+)*)>\s*(.*))$", name)
		if match:
			return int(match.group(2).replace(',', '').replace('.', '')), False
	return 1, True

deaths[['count', 'named']] = deaths['character'].apply(lambda x: pd.Series(count_and_name(x)))
kills[['killed', 'named']] = kills['victims'].apply(lambda x: pd.Series(count_and_name(x)))
edges[['count', 'named']] = edges['victim'].apply(lambda x: pd.Series(count_and_name(x)))

In [7]:
print('----- Deadliest Universes -----')
death_sum = deaths.groupby(by=['universe']).aggregate({'count':sum})
pd.DataFrame(death_sum.sort_values(by=['count'], ascending=False).reset_index().rename(columns={0:'deaths'}))

----- Deadliest Universes -----


,universe,count
0,Zero Escape,6000000221
1,Code Geass,35000058
2,Attack on Titan,20837951
3,Hunter × Hunter,5663910
4,Project Moon,380503
5,Hollow Knight,4962
6,Nasuverse,1311
7,Dragon's Dogma,934
8,Dark Souls,525
9,Elden Ring,511


In [8]:
def join_with_counts(series):
	seen = {}
	result = []
	for v in series:
		if v not in seen:
			seen[v] = 1
			result.append(v)
		else:
			seen[v] += 1
	return ", ".join(
		f"{v} (×{seen[v]})" if seen[v] > 1 else v
		for v in result
	)

In [9]:
kills = kills.groupby(by=['universe', 'character']).aggregate({'killed':sum, 'victims':join_with_counts})
kills.reset_index(inplace=True)
print('----- Deadliest Killer per Universe -----')
kills.drop('victims', axis=1).sort_values(by='killed', ascending=False).drop_duplicates('universe').reset_index(drop=True)

----- Deadliest Killer per Universe -----


,universe,character,killed
0,Zero Escape,Delta / Brother / Q / Zero II,6000000141
1,Code Geass,Lelouch vi Britannia / Lelouch Lamperouge / Ju...,35000020
2,Attack on Titan,Eren Jaeger,20000126
3,Hunter × Hunter,Meruem,538122
4,Project Moon,Carmen,380262
5,Hollow Knight,Hornet,2833
6,Nasuverse,Ritsuka Fujimaru,1015
7,Dragon's Dogma,The Arisen,912
8,Elden Ring,The Tarnished,422
9,Final Fantasy,Lightning / Claire Farron,226


In [10]:
deaths['character'] = deaths['character'].apply(lambda x: x if isinstance(x, list) else [x])
deaths_exploded = deaths.explode('character', ignore_index=True)
deaths_exploded = deaths_exploded[deaths_exploded['named'] == True]
deaths_count = deaths_exploded.groupby(['universe', 'character']).size().reset_index(name='deaths')

kills_count = kills.copy()
kills_count.drop(['victims'], axis=1, inplace=True)
kills_count.columns = ['universe', 'character', 'killed']

summary = pd.merge(deaths_count, kills_count, left_on=['universe', 'character'], right_on=['universe', 'character'], how='outer')
summary.fillna(0, inplace=True)
summary.sort_values(by=['universe', 'character'], inplace=True)

summary = summary.round(decimals=2).astype(object)
summary = summary.astype(str)
summary = summary.replace(to_replace = "\.0+$",value = "", regex = True)

In [11]:
deaths['resulting status'] = ''

for i in range(len(deaths)):
	type = str(deaths['type'][i]).split(' | ')

	conditions = ["determinant", "continuant"]
	if re.search('|'.join(conditions), str(type)):
		dependent = True
	else:
		dependent = False

	conditions = ["unavoidable"]
	if re.search('|'.join(conditions), str(type)):
		dependent = False

	conditions = ["revived", "reincarnated", "undone", "non-canon", "alternate"]
	if re.search('|'.join(conditions), str(type)):
		alive = True
	else:
		alive = False

	conditions = ["physical", "brain-dead", "undead"]
	if re.search('|'.join(conditions), str(type)):
		extant = True
	else:
		extant = False

	if alive:
		deaths['resulting status'][i] = 'alive'
	elif extant and not alive:
		deaths['resulting status'][i] = 'extant'
	elif dependent and not extant and not alive:
		deaths['resulting status'][i] = 'dependent'
	elif not dependent and not extant and not alive:
		deaths['resulting status'][i] = 'deceased'

In [12]:
priority = {"deceased": 0, "extant": 1, "dependent": 2, "alive": 3}

status = deaths.copy()
status["character"] = status["character"].apply(lambda x: x if isinstance(x, list) else [x])
status = status.explode("character", ignore_index=True)

status["priority"] = status["resulting status"].map(priority)

status = status.loc[status.groupby(["universe", "character"])["priority"].idxmin()]
status = status.drop(columns=["priority", "responsible party", "death", "type"]).reset_index(drop=True)
status.rename(columns={"resulting status": "last known status"}, inplace=True)

summary = pd.merge(summary, status, on=["universe", "character"], how="outer")
summary["last known status"] = summary["last known status"].fillna("alive")

### Generating the README file

In [13]:
progress = """### Terminology for Character Statuses

- **deceased** - the character stayed dead.
- **alive** - the character did not die or did not stay dead.
- - **revived**      - this death was nullified when the character was brought back from the dead.
- - **reincarnated** - this death was nullified when the character was reborn with their memories.
- - **undone**       - this death was nullified via time travel.
- - **non-canon**    - this death happened, but is not considered canon when it comes to the character's status.
- - **alternate**    - this death happened to a different version of the character, usually within an alternate universe.
- **dependent** - the character may or may not be deceased depending on certain factors.
- - **determinant** - this death happening depends upon a player's choice.
- - **continuant**  - this death exists but not in every adaptation of a story, and the character's canonical status is unspecified.
- - **unavoidable** - this death happens differently depending upon a choice but always happens nonetheless.
- **extant** - the character is arguably deceased yet still remains due to specific circumstances.
- - **physical**   - this death concerns only a character's body, leaving their mind or soul alive.
- - **brain-dead** - this death left the character only physically alive, with no brain activity possible.
- - **undead**     - this death turned the character into an undead being such as a zombie or a ghost.


### Status of Incomplete Entries

- Dr. Stone - missing characters revived via petrification
- Fate Grand Order - up to and including the **Third Singularity - Okeanos**
- Gantz - up to and including the **Dinosaur Alien Mission Arc**
- Hunter × Hunter - up to and including the **13th Hunter Chairman Election Arc**
- Limbus Company - up to and including **Intervallo VI: Spring Cultivation**
- Re:Zero - up to and including **Arc 3 - Return to the Royal Capital**


### Summary of Universes and Entries

"""

with open('README.md', 'w+') as f:
	f.write(progress)

In [14]:
deaths_summary = deaths.copy()
deaths_summary.rename(columns={'count': 'deaths'}, inplace=True)
deaths_summary = deaths_summary[deaths_summary['universe'] != '*']
deaths_summary = deaths_summary.drop(["character", "responsible party", "death", "type", "resulting status"], axis=1)
deaths_summary = deaths_summary.groupby(["universe", "entry"]).aggregate({"deaths": sum})
deaths_summary = deaths_summary.reset_index()

final_rows = []

grand_total = deaths_summary["deaths"].sum()
final_rows.append({
	"universe": "**\***",
	"entry": "**\***",
	"deaths": f"**{grand_total}**"
})

for universe, group in deaths_summary.groupby("universe"):
	universe_total = group["deaths"].sum()

	final_rows.append({
		"universe": f"**{universe}**",
		"entry": "**\***",
		"deaths": f"**{universe_total}**"
	})

	final_rows.extend(group.to_dict(orient='records'))

final_table = pd.DataFrame(final_rows)
deaths_table = final_table.to_markdown(tablefmt='github', index=False)

with open('README.md', 'a+') as f:
	f.write(deaths_table)

### Visualizing Death Links using PyVis

In [15]:
status = summary.copy()
status.drop(columns=["deaths", "killed"], inplace=True)
status.rename(columns={"last known status": "color"}, inplace=True)

colors = {"alive" : "#107a06",
		  "extant" : "#0990b5",
		  "dependent" : "#631da3",
		  "deceased" : "#730a1c"}

status["color"].replace(colors, inplace=True)
status["character"] += " [" + status["universe"] + "]"

In [16]:
edges['killer'] += " [" + edges['universe'] + "]"
edges.loc[edges['killer'].str.startswith(' ['), 'killer'] = ''
edges['victim'] += " [" + edges['universe'] + "]"

nodes = pd.DataFrame()
nodes['character'] = pd.concat([edges.loc[edges["named"], "victim"], edges['killer']]).drop_duplicates().sort_values().reset_index(drop=True)

nodes = pd.merge(nodes, status, left_on='character', right_on='character', how='left')
nodes.sort_values(by=['character', 'color'], inplace=True)

deaths["character"] = deaths["character"].apply(lambda x: x if isinstance(x, list) else [x])
deaths = deaths.explode("character", ignore_index=True)

descriptions = deaths.groupby(by=['universe', 'character']).aggregate({'death':'\n'.join})
descriptions.reset_index(inplace=True)
descriptions['character'] += " [" + descriptions['universe'] + "]"
descriptions.drop(columns=['universe'], inplace=True)

nodes = pd.merge(nodes, descriptions, left_on=['character'], right_on=['character'], how='left')

profiles = pd.DataFrame(columns=['universe', 'character', 'profile'])

In [17]:
def make_network(universe):
	if universe == '*':
		return

	net = Network(height="100vh", width="100%", bgcolor="#111111", directed=True, notebook=False)
	net.force_atlas_2based(overlap=1, damping=0.5)

	current_edges = edges[edges['universe'] == universe]
	draw_edges = current_edges[current_edges['named']].copy()
	current_nodes = nodes[nodes['character'].str.endswith('[' + universe + ']')]

	killer_total = (current_edges.groupby("killer")["count"].sum().to_dict())
	killer_unnamed = (current_edges[~current_edges["named"]].groupby("killer")["count"].sum().to_dict())

	sources, targets = draw_edges['killer'], draw_edges['victim']
	description = draw_edges['death']
	edge_data = zip(sources, targets, description)

	for i in range(len(current_nodes)):
		victim = current_nodes.iloc[i]['character']
		color = current_nodes.iloc[i]['color']
		label = victim.split(" [")[0]
		description = current_nodes.iloc[i]['death']
		title = label + "\n\nCharacter Profile :" + "\n\n\n"

		if description == description:
			value = description.count('\n') + 1
			if value != 1:
				plural = 's'
			else:
				plural = ''

			title += str(value) + " Death" + plural + " :\n\n" + description + "\n\n"
		else:
			title += ''

		if victim != '':
			font = f"15px baskerville {color} sans-serif"
			net.add_node(victim, label=label, title=title, color=color, font=font)

	for src, dst, des in edge_data:
		if src != '':
			net.add_edge(src, dst, title=des, color='#e8ac87', width=1, dashes=False)

	neighbor_map = net.get_adj_list()

	pair_counts = (draw_edges.groupby(["killer", "victim"])["count"].sum().to_dict())

	def format_neighbors_with_counts(node_id):
		raw = neighbor_map.get(node_id, [])
		seen = []
		for nbr in raw:
			if nbr not in seen:
				seen.append(nbr)

		formatted = []
		for nbr in seen:
			cnt = int(pair_counts.get((node_id, nbr), 1))
			formatted.append(f"{nbr} (×{cnt})" if cnt > 1 else nbr)

		return "\n".join(sorted(formatted))


	for node in net.nodes:
		nid = node['id']

		named_neighbors = neighbor_map.get(nid, [])
		unique_named = len(named_neighbors)

		total_victims = int(killer_total.get(nid, 0))
		unnamed_victims = int(killer_unnamed.get(nid, 0))

		if unique_named:
			plural = 's' if unique_named != 1 else ''
			node['title'] += f"{unique_named} Named Victim{plural} :\n\n"
			node['title'] += format_neighbors_with_counts(nid)
			node['title'] = re.sub(r' \[.*?\]', '', node['title'])

		if unnamed_victims:
			node["title"] += f"\n\n + {unnamed_victims} Unnamed Victims"

		node['value'] = max(total_victims, 1)

	net.inherit_edge_colors(False)
	net.set_edge_smooth('dynamic')

	net.save_graph('Networks/' + universe + '.html')

	for node in net.nodes:
		universe = node['id'].split(' [')[1].split(']')[0]
		character = node['id'].split(' [')[0]
		profile = node['title']
		row = [universe, character, profile]
		profiles.loc[len(profiles)] = row

In [18]:
for universe in kills['universe'].unique():
	make_network(universe)

### Exports

In [19]:
deaths = deaths[deaths['named']]
deaths.drop(columns=['count', 'named'], inplace=True)

summary.loc[summary['named'].isna(), 'named'] = True
summary = summary[summary['named']]
summary.drop(columns=['count', 'named'], inplace=True)

deaths.to_csv('deaths.tsv', sep='\t', index=False)
kills.to_csv('kills.tsv', sep='\t', index=False)
summary.to_csv('summary.tsv', sep='\t', index=False)